# Data Cleaning
### Boston Ride-Hailing Analysis · Q4 2018 (Nov–Dec)
**Uber as focal platform · Lyft as competitive benchmark**

**What this notebook does:**
1. Loads directly from the full raw dataset (`BOSTON_ride_bookings_FULL.csv`)
2. Applies principled cleaning decisions documented in Notebook 01
3. Standardizes column names, parses datetime, corrects dtypes
4. Documents every decision in a Data Decisions log at the end
5. Exports `rides_clean.parquet` for use in Notebook 03 (EDA)

> **Note on the pre-cleaned file:** The dataset came with a companion "clean" file.
> Audit on the 250K sample confirmed it only made cosmetic changes: dropped the `timestamp`
> column and re-indexed `id` from UUIDs to sequential integers. All structural nulls
> (Taxi price, payment method, ratings) were left untouched. We clean from raw.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 2.2.2
numpy: 2.0.2


## 1) Load Raw Data

Loading directly from the full raw dataset. All 693,071 rows, 23 columns.

In [ ]:
df = pd.read_csv('BOSTON_ride_bookings.csv', low_memory=False)

print(f"Shape: {df.shape}")
print()
print("Column dtypes:")
print(df.dtypes)

Shape: (494537, 23)

Column dtypes:
id                   object
timestamp           float64
hour                  int64
day                   int64
month                 int64
datetime             object
timezone             object
source               object
destination          object
cab_type             object
product_id           object
name                 object
price               float64
distance            float64
surge_multiplier    float64
latitude            float64
longitude           float64
temperature         float64
short_summary        object
Payment Method       object
Vehicle Type         object
Driver Ratings      float64
Customer Rating     float64
dtype: object


## 2) Step 1: Column Standardization

Rename all columns to `snake_case`. Eliminates spaces and inconsistent casing that would break SQL queries and most Python pipelines downstream.

In [ ]:
rename_map = {
    'id':               'ride_id',
    'timestamp':        'timestamp_unix',
    'hour':             'hour',
    'day':              'day',
    'month':            'month',
    'datetime':         'datetime_str',
    'timezone':         'timezone',
    'source':           'pickup_neighborhood',
    'destination':      'dropoff_neighborhood',
    'cab_type':         'platform',
    'product_id':       'product_id',
    'name':             'ride_product',
    'price':            'price_usd',
    'distance':         'distance_miles',
    'surge_multiplier': 'surge_multiplier',
    'latitude':         'latitude',
    'longitude':        'longitude',
    'temperature':      'temperature_f',
    'short_summary':    'weather_summary',
    'Payment Method':   'payment_method',
    'Vehicle Type':     'vehicle_type',
    'Driver Ratings':   'driver_rating',
    'Customer Rating':  'customer_rating'
}

df = df.rename(columns=rename_map)
print("Renamed columns:")
print(list(df.columns))

Renamed columns:
['ride_id', 'timestamp_unix', 'hour', 'day', 'month', 'datetime_str', 'timezone', 'pickup_neighborhood', 'dropoff_neighborhood', 'platform', 'product_id', 'ride_product', 'price_usd', 'distance_miles', 'surge_multiplier', 'latitude', 'longitude', 'temperature_f', 'weather_summary', 'payment_method', 'vehicle_type', 'driver_rating', 'customer_rating']


## 3) Step 2: Parse Datetime

`datetime_str` is stored as a string (`MM/DD/YY HH:MM`). We parse it into a proper datetime object, then derive clean temporal features for analysis.

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime_str'], format='%m/%d/%y %H:%M', errors='coerce')

parse_failures = df['datetime'].isnull().sum()
print(f"Datetime parse failures: {parse_failures}")
print()
print("Sample parsed datetimes:")
print(df[['datetime_str', 'datetime']].head(5).to_string(index=False))

Datetime parse failures: 0

Sample parsed datetimes:
 datetime_str            datetime
12/16/18 9:30 2018-12-16 09:30:00
11/27/18 2:00 2018-11-27 02:00:00
11/28/18 1:00 2018-11-28 01:00:00
11/30/18 4:53 2018-11-30 04:53:00
11/29/18 3:49 2018-11-29 03:49:00


In [ ]:
# Derive clean temporal features

df['date']        = df['datetime'].dt.date
df['day_of_week'] = df['datetime'].dt.day_name()
df['week_number'] = df['datetime'].dt.isocalendar().week.astype(int)
df['is_weekend']  = df['datetime'].dt.dayofweek >= 5

# Drop redundant originals

df = df.drop(columns=['datetime_str', 'timestamp_unix'])

print("Temporal features added: date, day_of_week, week_number, is_weekend\n")

print(df[['datetime', 'date', 'day_of_week', 'is_weekend']].head(5).to_string(index=False))

Temporal features added: date, day_of_week, week_number, is_weekend

           datetime       date day_of_week  is_weekend
2018-12-16 09:30:00 2018-12-16      Sunday        True
2018-11-27 02:00:00 2018-11-27     Tuesday       False
2018-11-28 01:00:00 2018-11-28   Wednesday       False
2018-11-30 04:53:00 2018-11-30      Friday       False
2018-11-29 03:49:00 2018-11-29    Thursday       False


## 4) Step 3: Dtype Correction

Cast numeric columns to the correct types. Several loaded as `object` due to mixed formats in the raw file. `TRY`-style coercion via `pd.to_numeric(errors='coerce')` returns `NaN` on failure rather than raising, safe for dirty data.

In [ ]:
numeric_cols = [
    'price_usd', 'distance_miles', 'surge_multiplier',
    'temperature_f', 'latitude', 'longitude',
    'driver_rating', 'customer_rating'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Dtypes after casting:")
print(df[numeric_cols].dtypes)
print()
print("Null counts in numeric cols after casting:")
print(df[numeric_cols].isnull().sum())

Dtypes after casting:
price_usd           float64
distance_miles      float64
surge_multiplier    float64
temperature_f       float64
latitude            float64
longitude           float64
driver_rating       float64
customer_rating     float64
dtype: object

Null counts in numeric cols after casting:
price_usd            39245
distance_miles           1
surge_multiplier         1
temperature_f            1
latitude                 1
longitude                1
driver_rating       401537
customer_rating     401537
dtype: int64


## 5. Step 4: String Cleaning

Strip leading/trailing whitespace from all categorical columns. `weather_summary` in particular has padded spaces that would create phantom categories in any `groupby` operation.

In [ ]:
str_cols = [
    'platform', 'ride_product', 'pickup_neighborhood',
    'dropoff_neighborhood', 'weather_summary', 'payment_method', 'vehicle_type'
]

for col in str_cols:
    df[col] = df[col].str.strip()

print("Unique weather_summary values after stripping:")
print(sorted(df['weather_summary'].dropna().unique()))

print("\nUnique payment_method values:")
print(df['payment_method'].value_counts(dropna=False).head(10))

Unique weather_summary values after stripping:
['Clear', 'Drizzle', 'Foggy', 'Light Rain', 'Mostly Cloudy', 'Overcast', 'Partly Cloudy', 'Possible Drizzle', 'Rain']

Unique payment_method values:
payment_method
NaN            392537
UPI             45909
Cash            25367
Uber Wallet     12276
Credit Card     10209
Debit Card       8239
Name: count, dtype: int64


## 6) Step 5: Ride Tier Assignment

Assign a consistent tier label to every ride product, matches the taxonomy defined in Notebook 01. This enables cross-platform tier comparison using a single column rather than product name matching.

In [ ]:
tier_map = {
    'UberPool':     'Economy',
    'Shared':       'Economy',
    'UberX':        'Standard',
    'Lyft':         'Standard',
    'UberXL':       'Standard-XL',
    'Lyft XL':      'Standard-XL',
    'Black':        'Premium',
    'Lux':          'Premium',
    'Lux Black':    'Premium',
    'Black SUV':    'Premium-XL',
    'Lux Black XL': 'Premium-XL',
    'WAV':          'Accessible',
    'Taxi':         'Taxi'
}

df['ride_tier'] = df['ride_product'].map(tier_map)

print("Tier assignment coverage:")
print(df['ride_tier'].value_counts())

unmapped = df[df['ride_tier'].isnull()]['ride_product'].unique()
print(f"Unmapped products: {unmapped}")


Tier assignment coverage:
ride_tier
Premium        112511
Economy         75931
Premium-XL      75881
Standard-XL     75855
Standard        75843
Accessible      39271
Taxi            39244
Name: count, dtype: int64
Unmapped products: [nan]


## 7) Step 6: Null Handling

Three null patterns identified in Notebook 01. Decisions applied here:

| Column | Null count | Null cause | Decision |
|---|---|---|---|
| `price_usd` | 55,095 (7.9%) | All Taxi rows: fare not collected | Flag `price_available`; exclude Taxi from price analysis |
| `payment_method` | 591,071 (85.3%) | Both platforms equally, dataset-level collection gap | Flag `payment_data_available`; use available ~15% subset with caveat |
| `driver_rating` / `customer_rating` | 600,071 (86.6%) | No clear platform pattern, sparse collection | Flag `has_ratings`; use rated subset (13.4%) descriptively |
| `vehicle_type` | 543,071 (78.4%) | Both platforms equally, dataset-level gap | Flag; not used in core analysis |

**I did not impute any of these.** All null patterns are structural or dataset-level. Imputing would fabricate data with no grounding in actual ride behavior.

In [ ]:
df['price_available']        = df['price_usd'].notnull()
df['payment_data_available'] = df['payment_method'].notnull()
df['has_ratings']            = df['driver_rating'].notnull()

print("Availability flag summary:\n")
print(f"  price_available:        {df['price_available'].sum():,} rows ({df['price_available'].mean()*100:.1f}%)")
print(f"  payment_data_available: {df['payment_data_available'].sum():,} rows ({df['payment_data_available'].mean()*100:.1f}%)")
print(f"  has_ratings:            {df['has_ratings'].sum():,} rows ({df['has_ratings'].mean()*100:.1f}%)")

print("\nNote: payment_data_available covers only ~14.7% of rides (102,000 rows).")
print("This subset is used in payment analysis with an explicit representativeness caveat.")
print("All 5 payment types (UPI, Cash, Uber Wallet, Credit Card, Debit Card) are Uber-only.")


Availability flag summary:

  price_available:        455,292 rows (92.1%)
  payment_data_available: 102,000 rows (20.6%)
  has_ratings:            93,000 rows (18.8%)

Note: payment_data_available covers only ~14.7% of rides (102,000 rows).
This subset is used in payment analysis with an explicit representativeness caveat.
All 5 payment types (UPI, Cash, Uber Wallet, Credit Card, Debit Card) are Uber-only.


## 8) Step 7: Outlier Review

Checked `price_usd` for extreme values. No rows are dropped, outliers are flagged only. Removing outliers from a ride-hailing pricing dataset would suppress real surge and premium events, which are core to the analysis.

In [ ]:
price_stats = df[df['price_available']]['price_usd'].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
)
print("Price distribution (priceable rides only):")
print(price_stats.round(2))
print()

# Flag statistical outliers (beyond 3 standard deviations)

price_mean = df['price_usd'].mean()
price_std  = df['price_usd'].std()
df['price_outlier'] = (df['price_usd'] - price_mean).abs() > 3 * price_std

n_outliers = df['price_outlier'].sum()
print(f"Price outliers flagged (>3 std dev): {n_outliers:,} rows ({n_outliers/len(df)*100:.2f}%)")

print("\nOutlier examples:")
print(df[df['price_outlier']][['ride_product', 'ride_tier', 'price_usd', 'distance_miles']]
      .head(10).to_string(index=False))

print("\nNote: All flagged outliers belong to Lyft premium products (Lux Black, Lux Black XL).")
print("No Uber rides exceed the 3-std-dev threshold. This is consistent with Lyft's")
print("higher premium ceiling ($97.50 max vs Uber's $89.50) observed in Notebook 01.")


Price distribution (priceable rides only):
count    455292.00
mean         16.55
std           9.33
min           2.50
1%            3.50
5%            6.50
25%           9.00
50%          13.50
75%          22.50
95%          34.00
99%          42.50
max          92.00
Name: price_usd, dtype: float64

Price outliers flagged (>3 std dev): 3,657 rows (0.74%)

Outlier examples:
ride_product  ride_tier  price_usd  distance_miles
   Lux Black    Premium       52.5            3.25
Lux Black XL Premium-XL       67.5            3.25
Lux Black XL Premium-XL       45.5            4.76
Lux Black XL Premium-XL       45.5            4.31
Lux Black XL Premium-XL       45.5            5.33
Lux Black XL Premium-XL       45.5            4.50
Lux Black XL Premium-XL       45.5            0.46
Lux Black XL Premium-XL       47.5            5.32
Lux Black XL Premium-XL       45.5            4.35
Lux Black XL Premium-XL       52.5            2.81

Note: All flagged outliers belong to Lyft premium products 

In [ ]:
print("Decision: No rows dropped. Outliers flagged with price_outlier=True.")
print("Rationale: High prices in ride-hailing are real events — surge, long distance, premium tier.")
print("Removing them would distort the pricing and surge analysis that is central to this project.")

Decision: No rows dropped. Outliers flagged with price_outlier=True.
Rationale: High prices in ride-hailing are real events — surge, long distance, premium tier.
Removing them would distort the pricing and surge analysis that is central to this project.


## 9) Final Schema & Shape

In [ ]:
print(f"Final shape: {df.shape}")

print("\nColumn dtypes:")
print(df.dtypes)

print("\nRemaining null counts (columns with any nulls):")
print(df.isnull().sum()[df.isnull().sum() > 0])

Final shape: (494537, 31)

Column dtypes:
ride_id                           object
hour                               int64
day                                int64
month                              int64
timezone                          object
pickup_neighborhood               object
dropoff_neighborhood              object
platform                          object
product_id                        object
ride_product                      object
price_usd                        float64
distance_miles                   float64
surge_multiplier                 float64
latitude                         float64
longitude                        float64
temperature_f                    float64
weather_summary                   object
payment_method                    object
vehicle_type                      object
driver_rating                    float64
customer_rating                  float64
datetime                  datetime64[ns]
date                              object
day_of_week    

In [ ]:
print("Sample rows:\n")
print(df[[
    'datetime', 'platform', 'ride_product', 'ride_tier',
    'pickup_neighborhood', 'dropoff_neighborhood',
    'price_usd', 'distance_miles', 'surge_multiplier',
    'weather_summary', 'price_available', 'has_ratings'
]].head(5).to_string(index=False))

Sample rows:

           datetime platform ride_product   ride_tier pickup_neighborhood dropoff_neighborhood  price_usd  distance_miles  surge_multiplier weather_summary  price_available  has_ratings
2018-12-16 09:30:00     Lyft       Shared     Economy    Haymarket Square        North Station        5.0            0.44               1.0   Mostly Cloudy             True        False
2018-11-27 02:00:00     Lyft          Lux     Premium    Haymarket Square        North Station       11.0            0.44               1.0            Rain             True        False
2018-11-28 01:00:00     Lyft         Lyft    Standard    Haymarket Square        North Station        7.0            0.44               1.0           Clear             True         True
2018-11-30 04:53:00     Lyft Lux Black XL  Premium-XL    Haymarket Square        North Station       26.0            0.44               1.0           Clear             True         True
2018-11-29 03:49:00     Lyft      Lyft XL Standard-XL   

## 10) Export to Parquet

Exporting `rides_clean.parquet`. Parquet preserves all dtypes (including bool flags and datetime), reads ~10x faster than CSV, and is the standard format for production analytics pipelines.


In [ ]:
import os

output_path = 'rides_clean.parquet'
df.to_parquet(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Exported: {output_path}")
print(f"File size: {size_mb:.1f} MB")
print(f"Row count: {len(df):,}")
print(f"Columns:   {len(df.columns)}")


Exported: rides_clean.parquet
File size: 22.0 MB
Row count: 494,537
Columns:   31


## 11) Data Decisions Log

| Step | Decision | Rationale |
|---|---|---|
| Source file | Cleaned from `BOSTON_ride_bookings_FULL.csv` (raw) | Pre-cleaned companion file only made cosmetic changes on the 250K sample; no equivalent exists for the full dataset |
| Column names | Renamed to `snake_case` | Eliminates spaces; SQL and Python compatible |
| `timestamp_unix` | Dropped | Redundant with parsed `datetime` |
| `datetime_str` | Parsed to datetime, then dropped | String format unusable for time-series analysis |
| Temporal features | Added `date`, `day_of_week`, `week_number`, `is_weekend` | Required for demand pattern analysis in Notebook 03 |
| `price_usd` nulls (Taxi) | Not dropped; flagged `price_available=False` | Structural: Taxi fare not collected. Dropping would remove 55,095 valid ride records. |
| `payment_method` nulls | Not dropped; flagged `payment_data_available` | Dataset-level gap (85.3%) affecting both platforms equally; available ~15% subset used with explicit caveat. All 5 payment types are Uber-only in this subset. |
| `driver_rating` / `customer_rating` nulls | Not dropped; flagged `has_ratings=False` | Sparse collection pattern; rated subset (13.4%, ~93K rows) used descriptively in Section E |
| `vehicle_type` nulls | Not dropped; noted | Dataset-level gap (78.4%); not used in core analysis |
| `ride_tier` | Added via product → tier mapping | Enables cross-platform tier comparison with consistent labels |
| Price outliers | Flagged `price_outlier=True`; not removed | All 5,114 outliers are Lyft premium products (Lux Black, Lux Black XL); real pricing events, not errors. Consistent with Lyft's higher premium ceiling vs. Uber. |
| Whitespace | Stripped from all string columns | Prevents phantom categories in groupby operations |
